# Table 1 descriptive statistics

## Import statements and read in data

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np

# Geopandas Dependency
import pyarrow

# Data reading
import requests
from io import BytesIO, StringIO

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
import leafmap.foliumap as leafmap
import plotly.express as px


# Model requirements
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier


# Model validation
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, precision_recall_curve, auc, precision_score, recall_score

In [2]:
# Read the origin time series file into a DataFrame
url_ts = "https://raw.githubusercontent.com/iansargent/nyc-subway-ridership-ml/main/Data/Clean/origin_ridership_time_series_CLEAN.csv"
origin = pd.read_csv(StringIO(requests.get(url_ts, verify=False).text))

In [3]:
display(origin.head())

,month,day_of_week,hour_of_day,origin_station_complex_id,origin_station_complex_name,origin_latitude,origin_longitude,origin_point,sum_estimated_average_ridership
0,1,Friday,0,1,"Astoria-Ditmars Blvd (N,W)",40.775036,-73.912034,POINT (-73.912034 40.775036),35.7473
1,1,Friday,1,1,"Astoria-Ditmars Blvd (N,W)",40.775036,-73.912034,POINT (-73.912034 40.775036),11.4975
2,1,Friday,2,1,"Astoria-Ditmars Blvd (N,W)",40.775036,-73.912034,POINT (-73.912034 40.775036),10.9980
3,1,Friday,3,1,"Astoria-Ditmars Blvd (N,W)",40.775036,-73.912034,POINT (-73.912034 40.775036),14.7534
4,1,Friday,4,1,"Astoria-Ditmars Blvd (N,W)",40.775036,-73.912034,POINT (-73.912034 40.775036),52.0023


In [4]:
origin.dtypes

month                                int64
day_of_week                         object
hour_of_day                          int64
origin_station_complex_id            int64
origin_station_complex_name         object
origin_latitude                    float64
origin_longitude                   float64
origin_point                        object
sum_estimated_average_ridership    float64
dtype: object

## Finding proportions for categorical variables and means for numeric variables

In [5]:
# sort by month
origin_sorted = origin.sort_values(by='month')

In [6]:
# Group by month and count occurrences
month_counts = origin_sorted['month'].value_counts()

# Step 3: Calculate percentages
total_month_count = month_counts.sum()
month_percentages = (month_counts / total_month_count) * 100

# Display results
print("Counts:")
print(month_counts)
print("\nPercentages:")
print(month_percentages)

Counts:
month
6     71197
12    71194
4     71191
10    71191
11    71173
8     71152
9     71137
3     71133
5     71066
2     70954
1     70827
7     70718
Name: count, dtype: int64

Percentages:
month
6     8.347315
12    8.346963
4     8.346611
10    8.346611
11    8.344501
8     8.342039
9     8.340280
3     8.339811
5     8.331956
2     8.318825
1     8.303935
7     8.291155
Name: count, dtype: float64


In [7]:
# proportions for day of the week

week_counts = origin['day_of_week'].value_counts()

# Calculate percentages
total_week_count = week_counts.sum()
week_percentages = (week_counts / total_week_count) * 100

# Display results
print("Counts:")
print(week_counts)
print("\nPercentages:")
print(week_percentages)

Counts:
day_of_week
Friday       121973
Tuesday      121971
Wednesday    121944
Thursday     121943
Monday       121913
Saturday     121615
Sunday       121574
Name: count, dtype: int64

Percentages:
day_of_week
Friday       14.300420
Tuesday      14.300185
Wednesday    14.297020
Thursday     14.296903
Monday       14.293385
Saturday     14.258447
Sunday       14.253640
Name: count, dtype: float64


In [8]:
# hour of day proportions

hour_counts = origin['hour_of_day'].value_counts()
hour_counts_df = hour_counts.reset_index().sort_values(by = 'hour_of_day')

# Calculate percentages
total_hour_count = hour_counts.sum()
hour_percentages = (hour_counts / total_hour_count) * 100
hour_percent_df = hour_percentages.reset_index().sort_values(by = 'hour_of_day')

# Display results
print("Counts:")
print(hour_counts_df)
print("\nPercentages:")
print(hour_percent_df)

Counts:
    hour_of_day  count
20            0  35465
22            1  35345
23            2  35309
21            3  35417
18            4  35549
14            5  35571
10            6  35574
9             7  35574
4             8  35577
11            9  35574
3            10  35578
12           11  35572
6            12  35576
5            13  35577
0            14  35580
7            15  35575
2            16  35579
1            17  35579
8            18  35575
15           19  35570
13           20  35571
16           21  35563
17           22  35554
19           23  35529

Percentages:
    hour_of_day     count
20            0  4.158005
22            1  4.143936
23            2  4.139716
21            3  4.152378
18            4  4.167854
14            5  4.170433
10            6  4.170785
9             7  4.170785
4             8  4.171137
11            9  4.170785
3            10  4.171254
12           11  4.170550
6            12  4.171019
5            13  4.171137
0            

In [9]:
# station proportions

station_counts = origin['origin_station_complex_name'].value_counts()

# Calculate percentages
total_station_count = station_counts.sum()
station_percentages = (station_counts / total_station_count) * 100

# Display results
print("Counts:")
print(station_counts)
print("\nPercentages:")
print(station_percentages)

Counts:
origin_station_complex_name
Astoria-Ditmars Blvd (N,W)    2016
Rector St (1)                 2016
Crown Hts-Utica Av (3,4)      2016
Kingston Av (3)               2016
Nostrand Av (3)               2016
                              ... 
Avenue X (F)                  1925
Beach 105 St (A,S)            1904
Neptune Av (F)                1901
Greenpoint Av (G)             1877
21 St (G)                     1823
Name: count, Length: 424, dtype: int64

Percentages:
origin_station_complex_name
Astoria-Ditmars Blvd (N,W)    0.236361
Rector St (1)                 0.236361
Crown Hts-Utica Av (3,4)      0.236361
Kingston Av (3)               0.236361
Nostrand Av (3)               0.236361
                                ...   
Avenue X (F)                  0.225692
Beach 105 St (A,S)            0.223230
Neptune Av (F)                0.222878
Greenpoint Av (G)             0.220064
21 St (G)                     0.213733
Name: count, Length: 424, dtype: float64


In [11]:
# counts just for most pop 5 and least pop 5
origin_st = station_counts.reset_index()
means = origin.groupby('origin_station_complex_name')['sum_estimated_average_ridership'].agg(['sum','mean','std'])

# Merge the DataFrames on the station column
descriptive_st = pd.merge(origin_st, means, on='origin_station_complex_name', how='inner')
# sort by ridership count to get mean/std for most popular and least popular 
least5_st = descriptive_st.sort_values(by = 'sum').head()
most5_st = descriptive_st.sort_values(by = 'sum', ascending=False).head()

display(least5_st)
display(most5_st)

,origin_station_complex_name,count,sum,mean,std
402,"Broad Channel (A,S)",1985,14257.9379,7.182840,8.047230
420,"Beach 105 St (A,S)",1904,16384.9678,8.605550,12.950915
412,"Beach 98 St (A,S)",1970,23178.3238,11.765647,12.773400
377,Beach 44 St (A),2011,34970.0174,17.389367,16.960013
394,Beach 36 St (A),1999,37420.5429,18.719631,18.440591


,origin_station_complex_name,count,sum,mean,std
132,"Times Sq-42 St (N,Q,R,W,S,1,2,3,7)/42 St (A,C,E)",2016,1.045785e+07,5187.423906,3451.312295
119,"Grand Central-42 St (S,4,5,6,7)",2016,7.733091e+06,3835.858764,3661.751237
98,"34 St-Herald Sq (B,D,F,M,N,Q,R,W)",2016,5.705197e+06,2829.958903,2349.290233
103,"14 St-Union Sq (L,N,Q,R,W,4,5,6)",2016,5.225462e+06,2591.994877,2198.687106
125,"Fulton St (A,C,J,Z,2,3,4,5)",2016,4.394671e+06,2179.896255,2169.393617


In [ ]:
# proportions just for most pop 5 and least pop 5
origin_stations = station_percentages.reset_index()
origin_stations.rename(columns = {'count': 'proportion'}, inplace = True)
means = origin.groupby('origin_station_complex_name')['sum_estimated_average_ridership'].agg(['sum','mean','std'])

# Merge the DataFrames on the station column
descriptive_station = pd.merge(origin_stations, means, on='origin_station_complex_name', how='inner')
# sort by ridership count to get mean/std for most popular and least popular 
least5_stations = descriptive_station.sort_values(by = 'sum').head()
most5_stations = descriptive_station.sort_values(by = 'sum', ascending=False).head()

display(least5_stations)
display(most5_stations)

In [ ]:
# Means and standard deviations for numeric columns
# will ignore the columns we know are actually categorical (month, hour, id)

print(origin['origin_latitude'].agg(['mean', 'std']).reset_index())
print(origin['origin_longitude'].agg(['mean', 'std']).reset_index())
print(origin['sum_estimated_average_ridership'].agg(['mean', 'std']).reset_index())

In [ ]:
# origin_point proportions

origin_counts = origin['origin_point'].value_counts()

# Step 3: Calculate percentages
total_origin_count = origin_counts.sum()
origin_percentages = (origin_counts / total_origin_count) * 100

# Display results
print("Counts:")
print(origin_counts)
print("\nPercentages:")
print(origin_percentages)

In [ ]:
# 5 least popular stations
# sort by ridership
least_pop_stations = origin.groupby('origin_station_complex_name')['sum_estimated_average_ridership'].sum().sort_values().head()
least_pop_stations

origin.groupby('origin_station_complex_name')['sum_estimated_average_ridership'].agg(['sum','mean','std']).sort_values(by = 'sum').head()

In [ ]:
# 5 most popular stations
origin.groupby('origin_station_complex_name')['sum_estimated_average_ridership'].agg(['sum','mean','std']).sort_values(by = 'sum', ascending=False).head()

## ridership mean and std by hour, day, month

In [ ]:
# mean and std of sum avg ridership by hour
origin.groupby('hour_of_day')['sum_estimated_average_ridership'].agg(['mean','std']).sort_values(by = 'hour_of_day')

In [ ]:
# order days of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
origin['day_of_week'] = pd.Categorical(origin['day_of_week'], categories=day_order, ordered=True)

In [ ]:
# mean and std of sum avg ridership by day

origin.groupby('day_of_week')['sum_estimated_average_ridership'].agg(['mean','std']).sort_values(by = 'day_of_week')

In [ ]:
# mean and std of sum avg ridership by month

origin.groupby('month')['sum_estimated_average_ridership'].agg(['mean','std']).sort_values(by = 'month')